# Bibliothèque de fiches catégories CIM-10 3-car

Génération massive des fiches markdown pour les ~2 054 catégories 3-caractères du référentiel (A18, M01, J18, R51…).

Architecture : `build_category_card` dans `src/recode_icd/cards.py`, sous-commande CLI `recode-icd cards build-categories`. Une fiche catégorie agrège les contenus de ses feuilles descendantes depuis le CSV maître.

Cas particulier validé : 465 catégories sont elles-mêmes des feuilles (R51, J22…), elles produisent une fiche sans section « Codes enfants directs ».

In [ ]:
from __future__ import annotations

import logging
import random
import time
from pathlib import Path

import polars as pl

from recode_icd.cards import (
    CATEGORY_FORMULATIONS_MAX,
    DEFAULT_SEED,
    build_categories_library,
    build_category_card,
)
from recode_icd.utils.loaders_dev import load_exploration_context

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

ctx = load_exploration_context(with_external=True)
assert ctx.flat is not None and ctx.merged is not None

pl.Config.set_fmt_str_lengths(120)
pl.Config.set_tbl_rows(30)

print(f"Plafond Formulations cat : {CATEGORY_FORMULATIONS_MAX}")

## 1 — Inventaire des catégories 3-car

In [ ]:
merged = ctx.merged
assert isinstance(merged, pl.DataFrame)
cat_3car = merged.filter(
    (pl.col("type") == "category") & (pl.col("code").str.len_chars() == 3)
).with_columns(
    pl.col("path").str.split("/").list.get(0).alias("chap")
)
print(f"Catégories 3-car : {cat_3car.height:,}")
print("\nDistribution par chapitre (top 15) :")
print(cat_3car.group_by("chap").len().sort("len", descending=True).head(15))

## 2 — Spot-check sur 5 catégories témoins

In [ ]:
rng = random.Random(DEFAULT_SEED)
for code in ("A18", "M01", "R51", "J18", "U07"):
    card = build_category_card(code, ctx, rng)
    print(f"--- {code} ({len(card)} chars) ---")
    print(card[:500] + "...\n" if len(card) > 500 else card + "\n")

## 3 — Mesure de performance sur 30 catégories

In [ ]:
_r = random.Random(0)
perf_sample = _r.sample(cat_3car["code"].to_list(), 30)
rng_perf = random.Random(DEFAULT_SEED)

t0 = time.perf_counter()
for c in perf_sample:
    _ = build_category_card(c, ctx, rng_perf)
elapsed = time.perf_counter() - t0
per_card_ms = (elapsed / 30) * 1000
projection_sec = (per_card_ms * cat_3car.height) / 1000

print(f"30 fiches en {elapsed:.2f}s → {per_card_ms:.1f} ms/fiche")
print(f"Projection {cat_3car.height:,} catégories : {projection_sec:.0f} s")

## 4 — Génération massive

In [ ]:
summary = build_categories_library(
    ctx=ctx,
    output_dir=Path("outputs/cards_library_categories"),
    progress=True,
)
print(f"\nCatégories total : {summary.n_codes_total:,}")
print(f"Fiches écrites   : {summary.n_written:,}")
print(f"Erreurs          : {summary.n_errors}")
print(f"Durée            : {summary.elapsed_seconds:.1f} s")
print(f"Index            : {summary.index_path}")

## 5 — Vérifications post-génération via `_index.csv`

In [ ]:
index = pl.read_csv(summary.index_path)
print(f"Lignes : {index.height:,}\n")

print("Distribution par chapitre (top 10) :")
print(index.group_by("chapter").len().sort("len", descending=True).head(10))

print("\nPrésence des sections :")
for col in ("has_perimetre", "has_exclusions", "has_formulations"):
    n = index.filter(pl.col(col)).height
    print(f"  {col:20s} : {n:5,} / {index.height:,} ({100*n/index.height:.1f}%)")

print("\nStats nb_chars + n_enfants :")
print(index.select(
    pl.col("nb_chars").min().alias("nbc_min"),
    pl.col("nb_chars").median().alias("nbc_med"),
    pl.col("nb_chars").max().alias("nbc_max"),
    pl.col("nb_chars").mean().alias("nbc_mean"),
    pl.col("n_enfants").max().alias("ne_max"),
    pl.col("n_enfants").mean().alias("ne_mean"),
))

print("\nCatégories = feuilles (n_enfants=0) :")
print(f"  {index.filter(pl.col('n_enfants') == 0).height} catégories")

print("\nTop 10 catégories les plus volumineuses :")
print(index.sort("nb_chars", descending=True).head(10).select("code", "chapter", "n_enfants", "nb_chars", "libelle"))

## 6 — Spot-check sur 5 fiches au hasard

In [ ]:
_r2 = random.Random(123)
spot_sample = _r2.sample(index["code"].to_list(), 5)
for c in spot_sample:
    fp = summary.output_dir / index.filter(pl.col("code") == c)["filepath"][0]
    print(f"=== {c} ({fp}) ===")
    print(fp.read_text()[:600])
    print("...\n")